In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os 

In [ ]:
os.getcwd()

In [ ]:
adata = sc.read_visium('/Users/shivaramakrishna/sc-analysis-hlca/visium')
adata.var_names_make_unique("var")
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

In [ ]:
adata.var_names_make_unique()

In [ ]:
adata

In [ ]:

adata.obs


In [ ]:
adata.var

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
sns.histplot(
    adata.obs["total_counts"][adata.obs["total_counts"] < 10000],
    kde=False,
    bins=40,
    ax=axs[1],
)
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=60, ax=axs[2])
sns.histplot(
    adata.obs["n_genes_by_counts"][adata.obs["n_genes_by_counts"] < 4000],
    kde=False,
    bins=60,
    ax=axs[3],
)

In [ ]:
sc.pp.filter_cells(adata, min_counts=5000)
sc.pp.filter_cells(adata, max_counts=35000)
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
print(f"#cells after MT filter: {adata.n_obs}")
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="clusters", directed=False, n_iterations=-1)


In [ ]:
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts", "clusters"], wspace=0.4)

In [ ]:
plt.rcParams["figure.figsize"] = (8, 8)
sc.pl.spatial(adata, img_key="hires",color=["total_counts", "n_genes_by_counts"])
#adata.uns

In [ ]:
sc.pl.spatial(adata, img_key="hires", color="clusters", size=1.5)


In [ ]:
sc.tl.rank_genes_groups(adata, "clusters", method="t-test")
sc.pl.rank_genes_groups_heatmap(adata, groups="1", n_genes=10, groupby="clusters")

In [ ]:
import scanpy as sc
import anndata as an
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanorama
import squidpy as sq

from pathlib import Path

In [ ]:
# load the pre-processed dataset
img = "BENG207/sc-analysis-hlca/sclc/spatial/cytassist_image.tiff"
img 
img = sq.im.ImageContainer("BENG207/sc-analysis-hlca/sclc/spatial/cytassist_image.tiff")
print(img)
print(adata.obsm["spatial"])
print(img.shape)  # Assuming 'img' is your image object
img.compute()

In [ ]:
sq.pl.spatial_scatter(adata, color="clusters")

In [ ]:
print(adata.uns)
scale = adata.uns['spatial']['CytAssist_11mm_FFPE_Human_Lung_Cancer']['scalefactors']['tissue_hires_scalef']
scale = 0.15580368

In [ ]:
sq.im.calculate_image_features(
    adata, img, features="summary", key_added="features", show_progress_bar=False, scale=scale
)


In [ ]:
# calculate features for different scales (higher value means more context)

feature_name = f"features_summary_scale{scale}"
sq.im.calculate_image_features(
        adata,
        img.compute(),
        features="summary",
        key_added=feature_name,
        n_jobs=4,
        scale=scale,
    )